# E9 — Summary 4-Panel (W2/W3/W4)

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


Reproduces the 4-panel layout from `notebooks/figures/13_summary_4panel_w1.png` for W2, W3, W4.

In [ ]:
def build_summary_panel(wk):
    wm = WINDOWS_META[wk]
    rows = []
    for day in wm['days']:
        for sym in [wm['front'], wm['back']]:
            fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
            if not fpath: continue
            df = pd.read_parquet(fpath[0], columns=['ts_event','bid_px_00','ask_px_00','bid_sz_00','ask_sz_00'])
            df['ts_event'] = pd.to_datetime(df['ts_event'])
            rth = ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) >= wm['rth_start_min']) &                   ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) <= wm['rth_end_min'])
            df = df[rth]
            ba = (df['ask_px_00']-df['bid_px_00'])/TICK
            sz = (df['bid_sz_00']+df['ask_sz_00'])/2
            rows.append(dict(day=day, symbol=sym,
                             mean_ba=ba.mean(), median_ba=ba.median(),
                             pct_1t=(ba==1).mean(), pct_2t=(ba==2).mean(), pct_3t=(ba>=3).mean(),
                             med_sz=sz.median()))
            del df; gc.collect()
    return pd.DataFrame(rows)

for wk in ['W2','W3','W4']:
    wm = WINDOWS_META[wk]
    df = build_summary_panel(wk)
    if df.empty:
        print(f'{wk}: no data found'); continue
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    x = range(len(wm['days']))
    for sym, style in [(wm['front'],'o-'),(wm['back'],'s--')]:
        sub = df[df['symbol']==sym].set_index('day').reindex(wm['days'])
        axes[0][0].plot(x, sub['mean_ba'],  style, label=sym)
        axes[0][1].plot(x, sub['median_ba'],style, label=sym)
        axes[1][0].plot(x, sub['pct_1t'],   style, label=sym)
        axes[1][1].plot(x, sub['med_sz'],   style, label=sym)
    for ax in axes.flatten():
        ax.set_xticks(list(x)); ax.set_xticklabels(wm['day_labels'], fontsize=8)
        ax.legend(fontsize=8)
    axes[0][0].set_title('Mean BA (ticks)'); axes[0][1].set_title('Median BA (ticks)')
    axes[1][0].set_title('Pct 1-tick quotes'); axes[1][1].set_title('Median Size (cts)')
    fig.suptitle(f'{wk}: {wm["front"]}→{wm["back"]} Bid-Ask Summary', fontsize=13)
    save_fig(fig, f'E9_summary_4panel_{wk.lower()}.png')
